# Problem statement

In this problem a program needs to perform a run-length-encoding (RLE) which is a common compression algorithm. RLE works by compressing consecutive objects representing the same data value. In this problem i will work only with text letters A-Z so for example 'AAAABBBCC'. Encoding this using RLE would yield '4A3B2C' as the numerical count represents the number of consequtive letters. 

The text can either be whitespaces or letters and if the text is for example '  jbjj  ' then the RLE is '2 jb2j2 '.


* What is the unknown? I run-length-encoded string
* What is the data? An original string with letters and whitespaces
* What is the condition? That the result returned i a string that represent the run-length-encoding of the string.


The second part of the statement asks to decode a text that is whenever there is a number before a character it then has to be written back in the original non-compressed form so '2A' becomes 'AA'. 

## The plan

### Decoding
This problem is about identifying numbers of consecutive symbols or letters. To do that i have to scan over the string. A naive solution as i see would.

1. Check if the text is empty, then return empty string.
2. Initialize a running sum to 0
3. Initialize a current object to the first element in the string
4. Initialize the output string as empty string
5. Start loop over the string character from the second element
6. If there is no second element just return the first character such as 'A', if that was the object to encode
7. If there is a second element, compare it with the first and if they are equal, increment running sum. Otherwise exchange the current element with the new one and add the first element as is to the output string. 
8. Moving to the next object in the string. Again if not exists just return output string. Otherwise compare against the now current element. 

### Encoding
To encode it's again about constructing a string based on the input string. If we encounter a number then we insert into the string that number of characters which comes after the number. The plan is to 

1. Initialize an empty string
2. Scan over the input string and check if the current character is an alpha. If it is then add it to the empty string.
3. If it's a number then figure out what the magnitude of the number is by either 1) run a while loop or 2) find the first regex digit pattern from the current position in the string to the end of the string. If 2) then i have to keep track of the current position in the scan which i guess would just be the index from the range() function. 

## Attempt

The first attempt i did was with a for loop. It works by first checking if the text is empty or of length 1. If so it returns either an empty string or the text itself. This is because if the string is only 1 element like 'A' or 'B' then we just return that. 

If the string is longer say 'AA' we scan over the string. It works as it is now, but might be able to optimize it. 

In [15]:
def encode(string):
    '''
    Function that compresses string using the run-length-encoding scheme
    '''

    if len(string) <= 1:
        return string

    # initializing variables 
    run_length_encoding = ''
    running_count = 1
    current_object = string[0]

    for char_num in range(1, len(string)):
        # scan over the string from the second element. 
        # compare that against the first
        if string[char_num] == current_object:
            # if equal then increment running count
            running_count += 1
            # if the character is the last one in the string then add the pattern current_count + current object to RLE.
            if char_num == len(string)-1:
                run_length_encoding = run_length_encoding + str(running_count) + current_object
                return run_length_encoding
            
        # if the character of the current object and char num is not equal
        # that means we have to halt any earlier compression step or just add the current object as is
        if string[char_num] != current_object:
            # then check if the running count is == 1
            if running_count == 1:
                # if so then add it the current object as is without number prefix
                run_length_encoding = run_length_encoding + current_object
            else:
                # if it's not then it means there has been consequctive letters therefore add the pattern
                run_length_encoding = run_length_encoding + str(running_count) + current_object

            # when done reset the running count to 1 and current object to the current object of the string. 
            running_count = 1
            current_object = string[char_num]
            # it's the last character then add it to the RLE
            # for example if the string goes 'AAB' then we have to add the B as well as it is. 
            # if instead it was 'AABB' then it will return from the first 'if' statement
            if char_num == len(string)-1:
                run_length_encoding = run_length_encoding + current_object

    return run_length_encoding

In [2]:
encode('zzz ZZ  zZ')

'3z 2Z2 zZ'

In [ ]:
def decode(string):
    '''
    Function that decodes run-length-encoding scheme 
    '''

    decode_string = ''
    char_num = 0
    while char_num < len(string):
        # check if current character is a digit
        if string[char_num].isdigit():
            # if it's a digit set num_pointer to char_num
            # also initialize digit as empty string
            num_pointer = char_num
            digit = ''
            while string[num_pointer].isdigit():
                # while num pointer is a digit
                # add to the digit the character at the num_pointer
                digit = digit + string[num_pointer]
                # increment num pointer
                num_pointer += 1    
            # once digit is constructed build the decode string as the decode string + (int(digit)*string[num_pointer])
            # num_pointer will be at exactly the letter after the digit
            decode_string = decode_string + int(digit) * string[num_pointer]
            # advance char_num to the letter after
            char_num += len(digit)+1
        else:
            decode_string = decode_string + string[char_num]
            char_num += 1

    return decode_string

In [ ]:
decode('12WB12W3B24WB')

In [4]:
decode(encode("zzz ZZ  zZ"))

'zzz ZZ  zZ'

## Looking back

### Reflection

* Was good that i spend time understanding the problem
* Was good that i made a plan

I tried to check now that i understand each step and there is some logical purpose behind it. This coding exercise was quite complex. The encoding part had some quirks that i didn't forsee in the planning phase. I found out that i needed to handle different logic when coming to the last character. 

For me now there is some logical reasoning about each step and i have tried to comment it in the solution. For example it makes sense that we need some running counter when we decode, one that tracks the number of consequctive letters. It also makes sense that we have a current object which we can compare new objects with when we can over the array - that comparison check helps us figure out if elements and consequtive. 


### Checking some community solutions
I saw a few community solutions. Some of them had similar approaches. I found one which uses a library called `itertools` which is think is a very useful one and like `re` i just haven't used it much. 


**Using itertools**

Using itertools actually makes it possible to concise the code quite alot. Particularly using the `groupby()` method. The group by method is able to group consecutive elements together which can be very useful for this problem. Imagine we can generate a list of groupings of characters then we can just loop through those list and check the length of each sublist and the character within it. If the length is greater than 1 we put the pattern of the len(list) + character.

In [ ]:
from itertools import groupby

def encode(string):
    rle = ''

    # creates a group of characters eg. 'AAABBBCCCAAA' becomes [['A','A','A], ['B', 'B', 'B']..]
    character_groupings = [list(group) for key, group in groupby(string)]

    # loops over each group
    for char_group in character_groupings:
        if len(char_group) > 1:
            rle += str(len(char_group)) + char_group[0]
        else:
            rle += char_group[0]

    return rle

In [46]:
def decode(string):
    count = ''
    decode_string = ''
    for k,g in groupby(string):
        if k.isdigit():
            count += k
        elif not count:
            decode_string += k
        else:
            decode_string += str(int(count)*k)
            count = ''

    return decode_string
